[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Headers and Content Types


## What you will be able to do

Read the headers a request sends and a response brings back, send headers of your own, ask a server
for the format you want, read a body the way its `Content-Type` says to, character set included, and
skip downloading a body you already have.


## The idea

### The problem

Every body so far was read with `response.json()`, because the code knew JSON was coming. A body does
not say what it is. A CSV table, a gateway's HTML page, JSON Lines and a JSON object all arrive as
bytes, and the bytes of a Norwegian place name read `Tromsø` in one character set and `TromsÃ¸` in
another. What a body is, and how its bytes become text, is written in the response's headers, and a
client that ignores them is guessing.

A request carries headers too, and they speak for the client. requests sends `Accept: */*`, which
says any format will do, so an API that can send the same table as JSON, CSV or HTML picks one for
it. It sends `User-Agent: python-requests`, which says nothing about who is calling, and some APIs
refuse a request that does not say.

Headers are also how a client and a server avoid repeating work. A client that already holds a
response can ask whether it has changed, and a server whose response has not changed answers
`304 Not Modified`, with no body to download.

### What headers are

> **Headers** are the `Name: value` lines of a request or a response, between its first line and
> its body. **Request headers** say who the client is and what it wants, as `User-Agent`, `Accept`
> and `If-None-Match` do. **Response headers** say what the body is and how to treat it, as
> `Content-Type`, `Content-Length` and `ETag` do. `Content-Type` holds a **media type**, such as
> `text/csv; charset=utf-8`: a type and a subtype, then parameters, of which `charset` names the
> character set of a text body.

### Why it works that way

- **A body is bytes.** HTTP carries bytes, so the headers are the one place where their meaning is
  written down. The same two bytes are `ø` in UTF-8 and `Ã¸` in ISO-8859-1.
- **Names ignore case, and values are text.** `content-type` and `Content-Type` are the same header,
  and every value is a string, a `Content-Length` of `73` included.
- **The client asks, and the server chooses.** `Accept` lists the formats a client takes, with a
  preference for each. The server sends the best one it can produce and names it in `Content-Type`,
  or answers `406 Not Acceptable`. This is **content negotiation**, and a server that does it sends
  `Vary: Accept`, telling caches that the response depends on that header.
- **A missing `charset` has a default, and it is rarely the right one.** HTTP's old default for text
  was ISO-8859-1, and requests still uses it for a `text/` type that arrives without a `charset`.
  JSON must be UTF-8, so requests decodes JSON as UTF-8 either way.
- **An `ETag` names one version of a body.** Sent back in `If-None-Match`, it asks the server to
  send the body only if the body has changed, and a `304` answer costs no body at all.

### Where you will meet this

GitHub's API refuses a request with no `User-Agent`, and asks clients to send
`Accept: application/vnd.github+json`, a media type of its own. OpenStreetMap's Nominatim geocoder
requires an identifying `User-Agent` in its usage policy. Open data portals publish tables as
`text/csv`, not always with a `charset`. Later in this guide, the **Authentication** notebook sends a
key in the `Authorization` header, the **Rate Limits** notebook reads a server's limits from its
response headers, and the **Sending Data** notebook sets `Content-Type` on a body the client sends.

### What this notebook covers

- The headers of a request and its response, and `/echo/headers`, which shows what a server received
- Headers of your own: a `User-Agent` that names the client, and a request id
- `Content-Type` taken apart into a media type and its parameters, and a body read by what it is
- `Accept`: one table as JSON, CSV or HTML, preferences with `q`, and `406 Not Acceptable`
- `charset`, and the default requests uses when a text body has none
- `ETag` and `If-None-Match`: a body you already have, answered with `304 Not Modified`
- A client that names itself, asks for a format, checks what arrived, and keeps what has not changed
- Five errors, from a `Content-Type` compared with `==` to an `ETag` sent back without its quotes

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

for accept in ["application/json", "text/csv"]:
    response = requests.get("http://127.0.0.1:8765/network/summary", headers={"Accept": accept}, timeout=10)
    print(response.status_code, response.headers["Content-Type"])
    print("   ", response.text[:49])
```

```
200 application/json
    [{"id": "bergen", "name": "Bergen", "local_name":
200 text/csv; charset=utf-8
    id,name,local_name,latitude,longitude,instruments
```

One address, two requests that differ only in `Accept`, and two bodies in two formats, each named in
its response's `Content-Type`.


## Setup

Nine imports, the last of them the practice API.

- `requests` sends every request, and its `headers=` adds headers to one
- `csv` reads the rows of a CSV body
- `io` wraps a string so that `csv` can read it like a file
- `json` reads a body of JSON Lines, a line at a time
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `open_meteo()` returns Open-Meteo's address, or the address of the practice API's recording of
  it when Open-Meteo is not answering

If Open-Meteo stops answering while you work through the notebook, run this cell again: it checks
again, and the Open-Meteo cells switch to the recording.


In [1]:
import csv
import importlib
import io
import json
import sys
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("The practice API is running at", BASE)
print("Open-Meteo's archive is at", OPEN_METEO)


The practice API is running at http://127.0.0.1:8765
Open-Meteo's archive is at https://archive-api.open-meteo.com/v1/archive


## Worked examples

### The headers of a request and its response

Every request and response in this guide has carried headers. Here are both sets for a station, as
requests keeps them:


In [2]:
response = requests.get(f"{BASE}/stations/tromso", timeout=10)

print("sent:    ", list(response.request.headers))
print("received:", dict(response.headers))
print(response.headers["content-length"], type(response.headers["Content-Length"]).__name__)


sent:     ['User-Agent', 'Accept-Encoding', 'Accept', 'Connection']
received: {'Server': 'PracticeAPI/1.0', 'Date': 'Sun, 01 Mar 2026 09:00:00 GMT', 'Content-Type': 'application/json', 'Content-Length': '73'}
73 str


requests sent four headers of its own, which the **Your First Request** notebook described, and only
their names are printed, because the value of `Accept-Encoding` depends on the compression libraries
installed. The practice API sent four back: `Server` names the software that answered, `Date` says
when, and `Content-Type` and `Content-Length` describe the body. The last line looked a header up in
lowercase, which works, and found a string: every header value is text, a length included.

### What the server received: /echo/headers

`/echo/headers` responds with the request headers it received, as `/echo` does with a query. Headers
given to `get` in `headers=` are sent along with the ones requests adds, and replace any of those
with the same name:


In [3]:
mine = {"User-Agent": "station-report/1.0 (reports@example.com)", "X-Request-Id": "report-0042"}
received = requests.get(f"{BASE}/echo/headers", headers=mine, timeout=10).json()["headers"]

for name in ["Host", "User-Agent", "Accept", "X-Request-Id"]:
    print(f"{name:<13} {received[name]}")


Host          127.0.0.1:8765
User-Agent    station-report/1.0 (reports@example.com)
Accept        */*
X-Request-Id  report-0042


`Host`, which names the server a request is for, was added by Python's HTTP connection, and `Accept`
is still requests' own `*/*`. The `User-Agent` is the one given: it names the program, its version,
and a way to reach its author, the form many APIs ask for, and `example.com` is a domain reserved
for examples like this one. `X-Request-Id` is a header of the program's own. Sent with every
request, it gives an API's operators an id to look for in their logs when a request is reported as a
problem.

### Content-Type: what a body is

Here is the `Content-Type` of five responses from earlier in this guide, one of them Open-Meteo's:


In [4]:
responses = {
    "a station": requests.get(f"{BASE}/stations/tromso", timeout=10),
    "the home page": requests.get(BASE, timeout=10),
    "the network export": requests.get(f"{BASE}/network/export", timeout=10),
    "a gateway failure": requests.get(f"{BASE}/status/502", timeout=10),
    "Open-Meteo's archive": requests.get(OPEN_METEO, timeout=30, params={
        "latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15", "end_date": "2025-01-17",
        "daily": "temperature_2m_mean", "models": "era5"}),
}

for label, response in responses.items():
    print(f"{label:<21} {response.headers['Content-Type']}")


a station             application/json
the home page         text/html; charset=utf-8
the network export    application/x-ndjson
a gateway failure     text/html; charset=utf-8
Open-Meteo's archive  application/json; charset=utf-8


A media type is a type and a subtype, as in `application/json`, which parameters may follow after
semicolons. The station and Open-Meteo's archive are both JSON, though their headers differ as text,
because Open-Meteo adds a `charset`. So a program compares the media type, not the whole header.
`media_type` splits a header into its two parts:


In [5]:
def media_type(content_type):
    """Split a Content-Type into its media type, lowercased, and a dictionary of its parameters."""
    kind, *parameters = content_type.split(";")
    pairs = (parameter.strip().split("=", 1) for parameter in parameters if "=" in parameter)
    return kind.strip().lower(), {name.lower(): value.strip('"') for name, value in pairs}


for label, response in responses.items():
    print(f"{label:<21} {media_type(response.headers['Content-Type'])}")


a station             ('application/json', {})
the home page         ('text/html', {'charset': 'utf-8'})
the network export    ('application/x-ndjson', {})
a gateway failure     ('text/html', {'charset': 'utf-8'})
Open-Meteo's archive  ('application/json', {'charset': 'utf-8'})


Media types ignore case, so `media_type` lowercases the type. With the type known, a program can read
any of these bodies the way it should be read:


In [6]:
def read_body(response):
    """The body as Python values, read the way its Content-Type says to read it."""
    kind, _ = media_type(response.headers.get("Content-Type", "application/octet-stream"))
    if kind == "application/json" or kind.endswith("+json"):
        return response.json()
    if kind == "application/x-ndjson":
        return [json.loads(line) for line in response.text.splitlines()]
    if kind == "text/csv":
        return list(csv.DictReader(io.StringIO(response.text)))
    if kind.startswith("text/"):
        return response.text
    return response.content


for label, response in responses.items():
    print(f"{label:<21} {type(read_body(response)).__name__}")


a station             dict
the home page         str
the network export    list
a gateway failure     str
Open-Meteo's archive  dict


JSON became Python values, JSON Lines a list with an item for each line, and HTML stayed text. The
test `kind.endswith("+json")` covers media types built on JSON, such as `application/problem+json`,
which many APIs use for error bodies, and GitHub's `application/vnd.github+json`. A body of any other
type comes back as bytes.

### Accept: asking for a format

`/network/summary` is a table of the four stations, and the practice API can send it three ways. The
request's `Accept` header decides which:


In [7]:
for accept in ["application/json", "text/csv", "text/html"]:
    response = requests.get(f"{BASE}/network/summary", headers={"Accept": accept}, timeout=10)
    print(f"{accept:<17} {response.status_code} {response.headers['Content-Type']:<25} Vary: {response.headers['Vary']}")


application/json  200 application/json          Vary: Accept
text/csv          200 text/csv; charset=utf-8   Vary: Accept
text/html         200 text/html; charset=utf-8  Vary: Accept


One address, three formats, and each response names its format in `Content-Type`. `Vary: Accept`
tells anything that keeps copies of responses, such as a proxy, that a copy made for one `Accept`
header must not be handed to a request with another.

A client can list several formats, each with a quality, `q`, from `0` to `1`, where a format with no
`q` counts as `1`. The server sends the format with the highest quality among those it can produce,
and where two tie, the one it prefers:


In [8]:
for accept in ["text/csv;q=0.5, text/html", "application/xml, text/*;q=0.8", "application/json;q=0, */*"]:
    response = requests.get(f"{BASE}/network/summary", headers={"Accept": accept}, timeout=10)
    print(f"{accept:<31} {response.status_code} {response.headers['Content-Type']}")


text/csv;q=0.5, text/html       200 text/html; charset=utf-8
application/xml, text/*;q=0.8   200 text/csv; charset=utf-8
application/json;q=0, */*       200 text/csv; charset=utf-8


The first prefers HTML to CSV. The second has no XML to offer, and `text/*` matches CSV and HTML
equally, so the practice API sent the one it lists first. In the third, `q=0` refuses JSON, and the
more specific `application/json` outranks `*/*`, which would otherwise have accepted it. When nothing
listed can be sent, the answer is `406 Not Acceptable`, with the formats that are available:


In [9]:
response = requests.get(f"{BASE}/network/summary", headers={"Accept": "application/xml"}, timeout=10)

print(response.status_code, response.reason)
print(response.json())


406 Not Acceptable
{'error': 'none of the formats in Accept is available', 'available': ['application/json', 'text/csv', 'text/html']}


Not every endpoint negotiates. The practice API's home page sends HTML whatever `Accept` says, as
many servers do, which is why a client checks `Content-Type` even after asking.

### charset: the characters in a text body

The table spells each station's name as it is spelled locally, and Tromsø has a letter outside ASCII.
requests takes the character set from `charset`, keeps it in `encoding`, and decodes `text` with it:


In [10]:
table = requests.get(f"{BASE}/network/summary", headers={"Accept": "text/csv"}, timeout=10)

print(table.encoding)
print(table.content.splitlines()[4])
print(table.text.splitlines()[4])


utf-8
b'tromso,Tromso,Troms\xc3\xb8,69.65,18.96,2'
tromso,Tromso,Tromsø,69.65,18.96,2


`content` is the bytes, where `ø` takes two, `\xc3\xb8`, and `text` is those bytes decoded as UTF-8.
`csv.DictReader` reads rows from text, and `io.StringIO` wraps the text so that it can be read like a
file:


In [11]:
rows = list(csv.DictReader(io.StringIO(table.text)))

print(rows[3])


{'id': 'tromso', 'name': 'Tromso', 'local_name': 'Tromsø', 'latitude': '69.65', 'longitude': '18.96', 'instruments': '2'}


Every value is text: CSV, like a query, has no types, so `latitude` stays `'69.65'` until code
converts it. Common errors shows the same table arriving without its `charset`.

### ETag and If-None-Match: a body you already have

The summary's responses carry an `ETag`, a name for the version of the body they hold. A client that
sends it back in `If-None-Match` asks for the body only if the body has changed:


In [12]:
first = requests.get(f"{BASE}/network/summary", timeout=10)
etag = first.headers["ETag"]
again = requests.get(f"{BASE}/network/summary", headers={"If-None-Match": etag}, timeout=10)

print("ETag: ", etag)
print("first:", first.status_code, first.reason, len(first.content), "bytes")
print("again:", again.status_code, again.reason, len(again.content), "bytes")


ETag:  "18f2fbb7de5a6cd0"
first: 200 OK 472 bytes
again: 304 Not Modified 0 bytes


The body had not changed, so the practice API answered `304 Not Modified` with no body, and the
client goes on using the bytes it already holds. The quotation marks belong to the `ETag`, which is
sent back exactly as it arrived, and each format has an `ETag` of its own, because each is a
different body. On an API with a rate limit, a conditional request saves more than bytes: GitHub's
API does not count a request answered with `304` against an authenticated client's limit.

### A client that names itself, asks, checks and keeps

Everything in this notebook, in one small client. `Client` names itself in every request, asks for
the format the caller wants, refuses a body in any other format, and keeps each body with its
`ETag`, so that a body it already holds costs a `304` instead of a download:


In [13]:
class Client:
    """Requests to one API that name the client, ask for a format, check it, and keep unchanged bodies."""

    def __init__(self, base, user_agent):
        self.base = base
        self.user_agent = user_agent
        self.kept = {}                                  # (path, format) -> (ETag, body)

    def get(self, path, wanted="application/json"):
        """The body at path in the wanted format, and whether it was fetched or kept."""
        headers = {"User-Agent": self.user_agent, "Accept": wanted}
        if (path, wanted) in self.kept:
            headers["If-None-Match"] = self.kept[path, wanted][0]
        response = requests.get(f"{self.base}{path}", headers=headers, timeout=10)
        if response.status_code == 304:
            return self.kept[path, wanted][1], "kept"
        response.raise_for_status()
        kind, _ = media_type(response.headers["Content-Type"])
        if kind != wanted:
            raise ValueError(f"asked {path} for {wanted}, and got {kind}")
        body = read_body(response)
        if "ETag" in response.headers:
            self.kept[path, wanted] = (response.headers["ETag"], body)
        return body, "fetched"


client = Client(BASE, "station-report/1.0 (reports@example.com)")

for path, wanted in [("/network/summary", "text/csv"), ("/network/summary", "text/csv"),
                     ("/network/summary", "application/json"), ("/stations/tromso", "application/json")]:
    body, how = client.get(path, wanted)
    print(f"{how:<8} {path:<18} {wanted:<17} a {type(body).__name__} of {len(body)}")

try:
    client.get("/", "application/json")
except ValueError as error:
    print("refused:", error)


fetched  /network/summary   text/csv          a list of 4
kept     /network/summary   text/csv          a list of 4
fetched  /network/summary   application/json  a list of 4
fetched  /stations/tromso   application/json  a dict of 4
refused: asked / for application/json, and got text/html


### Where each part came from

| In the client | What it relies on | The section that showed it |
|---|---|---|
| `"User-Agent": self.user_agent` | a header of the program's own, replacing requests' | What the server received: /echo/headers |
| `"Accept": wanted` | asking for a format | Accept: asking for a format |
| `media_type(response.headers["Content-Type"])` | comparing the media type, not the whole header | Content-Type: what a body is |
| `read_body(response)` | a body read by its media type and decoded with its charset | Content-Type: what a body is |
| `If-None-Match`, then `status_code == 304` | a body already held, not sent again | ETag and If-None-Match: a body you already have |

The second request for the CSV table cost a `304`. The station would be fetched in full every time
it was asked for, because its responses carry no `ETag` to send back.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/08-headers-and-content-types-solutions.ipynb).

**1.** Send `/echo/headers` a header `Accept-Language` set to `nb, en;q=0.8`, and print the value the
server received.


In [14]:
# your code here


**2.** Print every header of the response from `/network/export`, one to a line, as `Name: value`.


In [15]:
# your code here


**3.** Ask `/network/summary` for `text/html`, and print the response's `Content-Type` and the first
three lines of its body.


In [16]:
# your code here


**4.** Use `media_type` to print the `charset` of the home page's response and of a station's
response, printing `None` for a response that has none.


In [17]:
# your code here


**5.** Print the `ETag` of `/network/summary` as JSON and as CSV. Then ask for JSON again, sending the
CSV's `ETag` in `If-None-Match`, and print the status code.


In [18]:
# your code here


**6.** Write `chosen(accept)`, which sends `/network/summary` the given `Accept` header and returns
the media type of the response, or `None` for a `406`. Print what it returns for `text/plain`,
`text/*`, and `*/*;q=0.1, text/html;q=0.2`.


In [19]:
# your code here


## Common errors

### No error, and JSON taken for something else: a Content-Type compared with ==


In [20]:
archive = responses["Open-Meteo's archive"]

if archive.headers["Content-Type"] == "application/json":
    print("reading JSON")
else:
    print("not JSON, so not read:", archive.headers["Content-Type"])


not JSON, so not read: application/json; charset=utf-8


The body was JSON. The header was not the string `application/json`, because it carries a `charset`,
so the comparison failed, and the program skipped a body it could read. Compare the media type:


In [21]:
print("reading JSON" if media_type(archive.headers["Content-Type"])[0] == "application/json" else "not JSON")


reading JSON


### No error, and TromsÃ¸: a text body with no charset


In [22]:
legacy = requests.get(f"{BASE}/network/summary.csv", timeout=10)

print(legacy.headers["Content-Type"], "->", legacy.encoding)
print(legacy.text.splitlines()[4])


text/csv -> ISO-8859-1
tromso,Tromso,TromsÃ¸,69.65,18.96,2


`/network/summary.csv`, an older address for the same table, sends `text/csv` with no `charset`, and
requests fell back on ISO-8859-1, HTTP's old default for text. In ISO-8859-1 every byte is a
character of its own, so the two bytes of `ø` became two characters, `Ã¸`. Nothing raised, because the
text is valid, only wrong. When the documentation, or the bytes, show the real character set, set
`encoding` before reading `text`:


In [23]:
legacy.encoding = "utf-8"

print(legacy.text.splitlines()[4])


tromso,Tromso,Tromsø,69.65,18.96,2


`legacy.apparent_encoding` guesses a character set from the bytes themselves, which helps when there
is nothing else to go on, but it is a guess, and the shorter the body, the less it has to work with.

### InvalidHeader: Header part (42) from ('X-Request-Id', 42) must be of type str or bytes, not <class 'int'>


In [24]:
requests.get(f"{BASE}/echo/headers", headers={"X-Request-Id": 42}, timeout=10)


InvalidHeader: Header part (42) from ('X-Request-Id', 42) must be of type str or bytes, not <class 'int'>

A header value travels as text, and requests does not decide how a number should be written, so it
raises before anything is sent. Convert the value yourself:


In [25]:
received = requests.get(f"{BASE}/echo/headers", headers={"X-Request-Id": str(42)}, timeout=10).json()["headers"]

print(received["X-Request-Id"])


42


### HTTPError: 406 Client Error: Not Acceptable for url: http://127.0.0.1:8765/network/summary


In [26]:
response = requests.get(f"{BASE}/network/summary", headers={"Accept": "application/xml"}, timeout=10)
response.raise_for_status()


HTTPError: 406 Client Error: Not Acceptable for url: http://127.0.0.1:8765/network/summary

The request was sound, and the practice API has no XML to send. A `406` is a `4xx`, so the request
has to change, as the **Status Codes** notebook put it, and this body says to what:


In [27]:
available = response.json()["available"]
response = requests.get(f"{BASE}/network/summary", headers={"Accept": available[1]}, timeout=10)

print(available, "->", response.status_code, response.headers["Content-Type"])


['application/json', 'text/csv', 'text/html'] -> 200 text/csv; charset=utf-8


### No error, and no 304: an ETag sent back without its quotes


In [28]:
unquoted = first.headers["ETag"].strip('"')
response = requests.get(f"{BASE}/network/summary", headers={"If-None-Match": unquoted}, timeout=10)

print(unquoted, "->", response.status_code, len(response.content), "bytes")


18f2fbb7de5a6cd0 -> 200 472 bytes


The quotation marks are part of the `ETag`, and without them it is a different value, which names no
version of the body, so the whole body came again. Nothing failed, which makes this easy to miss:
the client worked, and saved nothing. Send the `ETag` back exactly as it arrived:


In [29]:
response = requests.get(f"{BASE}/network/summary", headers={"If-None-Match": first.headers["ETag"]}, timeout=10)

print(first.headers["ETag"], "->", response.status_code, len(response.content), "bytes")


"18f2fbb7de5a6cd0" -> 304 0 bytes


## Recap

- Request headers say who the client is and what it wants, and response headers say what the body
  is. Names ignore case, and every value is a string.
- `headers=` sends headers of your own, replacing requests' own of the same name; name a program in
  `User-Agent`.
- Read `Content-Type` as a media type and its parameters, compare the media type, and read the body
  the way that type says to.
- `Accept` asks for formats, with `q` for preferences; the server sends one and names it in
  `Content-Type`, or answers `406`.
- requests decodes a text body that has no `charset` as ISO-8859-1; set `encoding` when the real
  character set is known.
- Send an `ETag` back in `If-None-Match`, exactly as it arrived; a `304` means the body already held
  is current.


## What is next

The **Authentication** notebook. Every request here was answered for anyone who asked. That notebook
sends the header that says who is asking, with a key or a token in `Authorization`, and meets the
`401` and `403` responses the **Status Codes** notebook named.


---

&#8592; **Previous:** [Schemas and Validation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/07-schemas-and-validation.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Authentication](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/09-authentication.ipynb) &#8594;
